## 05.02节练习参考答案

### 环境准备

In [1]:
import os, sys
os.environ["TILE_FWK_DEVICE_ID"] = "0"

import logging
import sys
import warnings
logging.disable(logging.INFO)
warnings.filterwarnings("ignore", category=UserWarning)
sys.stderr.flush()
os.dup2(os.open(os.devnull, os.O_WRONLY), 2)

# 本 notebook 位于 answers/ 子目录下，src 包在其上一级目录。
# 把上层目录加入 sys.path，才能 from src.pto_layers import ...
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

import pypto
import torch
import torch_npu
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import math

from src.pto_layers import PyPTOLinear, PyPTOReLU

本节的解答思路参考了 [《动手学深度学习》习题解答](https://datawhalechina.github.io/d2l-ai-solutions-manual/#/ch05/ch05) ，并在此基础上补充了 PyPTO 的实现。  

### 练习5.2.1
查看初始化模块文档以了解不同的初始化方法。

**解答：**
通过查看深度学习框架文档，有以下初始化方法 （参考链接：https://pytorch.org/docs/stable/nn.init.html ）
* `torch.nn.init.uniform_(tensor, a=0.0, b=1.0)`：从均匀分布 *U(a,b)* 中提取填充输入张量。
* `torch.nn.init.normal_(tensor, mean=0.0, std=1.0)`：从正态分布 *N(mean,std²)* 中提取填充输入张量。
* `torch.nn.init.constant_(tensor, val)`：以一确定数值初始化张量。
* `torch.nn.init.ones_(tensor)`：用标量值 1 填充输入张量。
* `torch.nn.init.zeros_(tensor)`：用标量值 0 填充输入张量。
* `torch.nn.init.eye_(tensor)`：用单位矩阵填充二维输入张量
* `torch.nn.init.xavier_uniform_(tensor, gain=1.0)`：从均匀分布 *U(−a,a)* 中采样，初始化输入张量，其中 a 的值由如下公式确定
$$
a = gain * \sqrt{\frac{6}{fan_{in} + fan_{out}}}
$$
其中 gain 的取值如下表所示

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr><th style="text-align: left;">非线性函数</th><th style="text-align: left;">gain 值</th></tr>
</thead>
<tbody>
<tr><td>Linear/Identity</td><td></td></tr>
<tr><td>Conv1D</td><td>$1$</td></tr>
<tr><td>Conv2D</td><td>$1$</td></tr>
<tr><td>Conv3D</td><td>$1$</td></tr>
<tr><td>Sigmoid</td><td>$1$</td></tr>
<tr><td>Tanh</td><td>$\dfrac{5}{3}$</td></tr>
<tr><td>ReLU</td><td>$\sqrt{2}$</td></tr>
<tr><td>Leaky ReLU</td><td>$\sqrt{\dfrac{2}{1 + negative\_slope^2}}$</td></tr>
<tr><td>SELU</td><td>$1$ (adaptive)</td></tr>
</tbody>
</table>

* `torch.nn.init.xavier_normal_(tensor, gain=1.0)`:从正态分布 *N(0,std²)* 中采样，初始化输入张量，其中 std 值由下式确定：
$$
a = gain * \sqrt{\frac{2}{fan_{in} + fan_{out}}}
$$
* `torch.nn.init.kaiming_uniform_(tensor, a=0, mode='fan_in', nonlinearity='leaky_relu')`:服从均匀分布 *U(−bound,bound)*，其中 bound 值由下式确定
$$
bound = gain \cdot \sqrt{\dfrac{3}{fan_{mode}}}
$$
* `torch.nn.init.kaiming_normal_(tensor, a=0, mode='fan_in', nonlinearity='leaky_relu')`:服从从正态分布 *N(0,std 2)* 中采样，其中 std 值由下式确定
$$
std = \frac{gain}{\sqrt{fan_{mode}}}
$$
* `torch.nn.init.trunc_normal_(tensor, mean=0.0, std=1.0, a=- 2.0, b=2.0)`:用从截断的正态分布中提取的值填充输入张量。这些值实际上是从正态分布 *N(mean,std²)* 中提取的。
* `torch.nn.init.sparse_(tensor, sparsity, std=0.01)`：将 2D 输入张量填充为稀疏矩阵，其中非零元素将从正态分布 *N(0,0.01)* 中提取。

### 练习5.2.2
构建包含共享参数层的多层感知机并对其进行训练。在训练过程中，观察模型各层的参数和梯度。

**解答：**
在训练过程中，我们每个 epoch 都打印了每层的参数和梯度。可以看到 shared_fc 层的参数和梯度都是相同的，因为它们共享同一个参数。

In [2]:
# 模型参数
input_size = 4
hidden_size = 8
output_size = 4
lr = 0.01
epochs = 2

# 构建带有共享参数层的多层感知机
shared_fc = nn.Linear(hidden_size, hidden_size)
MLP = nn.Sequential(nn.Linear(input_size, hidden_size), nn.ReLU(),
                    shared_fc, nn.ReLU(),
                    shared_fc, nn.ReLU(),
                    nn.Linear(hidden_size, output_size)
)

# 训练数据
X = torch.randn(1, input_size)
Y = torch.randn(1, output_size)
# 优化器
optimizer = optim.SGD(MLP.parameters(), lr=lr)
# 训练模型
for epoch in range(epochs):
    # 前向传播和计算损失
    Y_pred = MLP(X)
    loss = nn.functional.mse_loss(Y_pred, Y)
    # 反向传播和更新梯度
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    # 打印每层参数和梯度的范数（精简输出，便于观察训练中的变化）
    print(f"--- Epoch {epoch} ---")
    for name, param in MLP.named_parameters():
        print(f"{name}: shape={tuple(param.shape)}, "
              f"param_norm={param.data.norm().item():.4f}, grad_norm={param.grad.norm().item():.4f}")
    print(f"Loss: {loss.item():.4f}")

# 验证共享参数层指向同一个模块（named_parameters 中 2.* 只出现一次）
print(f"\n共享参数验证 (MLP[2] is MLP[4]): {MLP[2] is MLP[4]}")

--- Epoch 0 ---
0.weight: shape=(8, 4), param_norm=1.4194, grad_norm=0.0548
0.bias: shape=(8,), param_norm=1.0518, grad_norm=0.0430
2.weight: shape=(8, 8), param_norm=1.9403, grad_norm=0.0889
2.bias: shape=(8,), param_norm=0.6890, grad_norm=0.1704
6.weight: shape=(4, 8), param_norm=1.2416, grad_norm=0.2823
6.bias: shape=(4,), param_norm=0.4305, grad_norm=0.6919
Loss: 0.4787
--- Epoch 1 ---
0.weight: shape=(8, 4), param_norm=1.4194, grad_norm=0.0553
0.bias: shape=(8,), param_norm=1.0519, grad_norm=0.0434
2.weight: shape=(8, 8), param_norm=1.9404, grad_norm=0.0899
2.bias: shape=(8,), param_norm=0.6896, grad_norm=0.1716
6.weight: shape=(4, 8), param_norm=1.2420, grad_norm=0.2831
6.bias: shape=(4,), param_norm=0.4255, grad_norm=0.6876
Loss: 0.4728

共享参数验证 (MLP[2] is MLP[4]): True


**PyPTO 版**

In [3]:
# 模型参数
input_size = 4
hidden_size = 8
output_size = 4
lr = 0.01
epochs = 2

# 构建带有共享参数层的多层感知机
shared_fc = PyPTOLinear(hidden_size, hidden_size)
MLP = nn.Sequential(PyPTOLinear(input_size, hidden_size), PyPTOReLU(),
                    shared_fc, PyPTOReLU(),
                    shared_fc, PyPTOReLU(),
                    PyPTOLinear(hidden_size, output_size)
)

# 训练数据
X = torch.randn(1, input_size).npu()
Y = torch.randn(1, output_size).npu()
# 优化器
optimizer = optim.SGD(MLP.parameters(), lr=lr)
# 训练模型
for epoch in range(epochs):
    # 前向传播和计算损失
    Y_pred = MLP(X)
    loss = nn.functional.mse_loss(Y_pred, Y)
    # 反向传播和更新梯度
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    # 打印每层参数和梯度的范数（精简输出，便于观察训练中的变化）
    print(f"--- Epoch {epoch} ---")
    for name, param in MLP.named_parameters():
        print(f"{name}: shape={tuple(param.shape)}, "
              f"param_norm={param.data.norm().item():.4f}, grad_norm={param.grad.norm().item():.4f}")
    print(f"Loss: {loss.item():.4f}")

# 验证共享参数层指向同一个模块（named_parameters 中 2.* 只出现一次）
print(f"\n共享参数验证 (MLP[2] is MLP[4]): {MLP[2] is MLP[4]}")

--- Epoch 0 ---
0.weight: shape=(8, 4), param_norm=1.5258, grad_norm=0.1061
0.bias: shape=(8,), param_norm=0.7883, grad_norm=0.0494
2.weight: shape=(8, 8), param_norm=1.6345, grad_norm=0.3825
2.bias: shape=(8,), param_norm=0.3531, grad_norm=0.4584
6.weight: shape=(4, 8), param_norm=1.2835, grad_norm=0.3539
6.bias: shape=(4,), param_norm=0.4712, grad_norm=1.1466
Loss: 1.3147
--- Epoch 1 ---
0.weight: shape=(8, 4), param_norm=1.5258, grad_norm=0.1033
0.bias: shape=(8,), param_norm=0.7882, grad_norm=0.0480
2.weight: shape=(8, 8), param_norm=1.6343, grad_norm=0.3757
2.bias: shape=(8,), param_norm=0.3528, grad_norm=0.4519
6.weight: shape=(4, 8), param_norm=1.2833, grad_norm=0.3445
6.bias: shape=(4,), param_norm=0.4672, grad_norm=1.1387
Loss: 1.2967

共享参数验证 (MLP[2] is MLP[4]): True


### 练习5.2.3
为什么共享参数是个好主意？

**解答：**
1. 节约内存：共享参数可以减少模型中需要存储的参数数量，从而减少内存占用。

2. 加速收敛：共享参数可以让模型更加稳定，加速收敛。

3. 提高泛化能力：共享参数可以帮助模型更好地捕捉数据中的共性，提高模型的泛化能力。

4. 加强模型的可解释性：共享参数可以让模型更加简洁明了，加强模型的可解释性。